# Book Text Partitioner
Randomly samples 200 partitions of 100 words each from each book.
Labels each partition as `bookName_partition_a`, `bookName_partition_b`, etc.
Outputs `partitions.csv` consumed by the model training notebook.

In [1]:
import re
import random
import string
import pandas as pd
from pathlib import Path

## Configuration
All five project books are pre-filled. Update `BOOKS_DIR` if your `.txt` files
live in a different folder. The `GENRE_MAP` links each filename stem to its genre
and is written into the output CSV so the training notebook has labels ready.

In [2]:
# ── USER CONFIGURATION ──────────────────────────────────────────────────────
BOOKS_DIR           = "KnownBooks"   # folder containing the _clean.txt files
PARTITIONS_PER_BOOK = 200
WORDS_PER_PARTITION = 400
RANDOM_SEED         = 42    # set to None for non-reproducible runs
OUTPUT_CSV          = "TestingData/partitions.csv"

# Filename stem -> genre label (25 books, 5 per genre)
GENRE_MAP = {
    # biography
    "AutobiographyOfBenjaminFranklin":  "biography",
    "LifeOfKingEdwardVII":              "biography",
    "NarrativeOfFrederickDouglass":     "biography",
    "StoryOfMyLife":                    "biography",
    "AutobiographyOfCharlesDarwin":     "biography",
    # fantasy
    "TheWonderfulWizardOfOz":           "fantasy",
    "AliceInWonderland":                "fantasy",
    "ThroughTheLookingGlass":           "fantasy",
    "GrimmsFairyTales":                 "fantasy",
    "GulliversTravels":                 "fantasy",
    # horror
    "Frankenstein":                     "horror",
    "CallOfCthulu":                     "horror",
    "Dracula":                          "horror",
    "TalesOfPoe":                       "horror",
    "DrJekyllAndMrHyde":                "horror",
    # romance
    "PrideAndPrejudice":                "romance",
    "RomeoAndJuliet":                   "romance",
    "JaneEyre":                         "romance",
    "WutheringHeights":                 "romance",
    "SenseAndSensibility":              "romance",
    # sci-fi
    "OnTheTrailOfTheSpacePirates":      "sci-fi",
    "PlagueShip":                       "sci-fi",
    "TheWarOfTheWorlds":                "sci-fi",
    "TheTimeMachine":                   "sci-fi",
    "TwentyThousandLeagues":            "sci-fi",
}

# Build file list from the genre map (expects <stem>_clean.txt)
BOOK_FILES = [f"{stem}_clean.txt" for stem in GENRE_MAP]
# ────────────────────────────────────────────────────────────────────────────

## Helper Functions

In [3]:
def load_and_clean(filepath: str) -> list[str]:
    """
    Read a pre-cleaned .txt file and return a list of whitespace tokens.
    Falls back to latin-1 if UTF-8 fails.
    """
    path = Path(BOOKS_DIR) / filepath
    try:
        raw = path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        raw = path.read_text(encoding="latin-1")

    raw = re.sub(r'\s+', ' ', raw).strip()
    return [t for t in re.split(r'\s+', raw) if t]


def book_label(filepath: str) -> str:
    """Derive a clean label from the filename stem (strips _clean suffix)."""
    stem = Path(filepath).stem
    stem = re.sub(r'_clean$', '', stem, flags=re.IGNORECASE)
    return re.sub(r'[\s_]+', '', stem)


def int_to_label(n: int) -> str:
    """Convert 0-based integer to alphabetic label: 0->'a', 26->'aa', etc."""
    letters = string.ascii_lowercase
    result, n = [], n + 1
    while n > 0:
        n, r = divmod(n - 1, 26)
        result.append(letters[r])
    return ''.join(reversed(result))


def sample_partitions(tokens, book_name, genre, n_partitions, words_per_partition, rng):
    """Draw n random windows of fixed width from the token list."""
    max_start = len(tokens) - words_per_partition
    if max_start < 1:
        raise ValueError(f"'{book_name}' too short for {words_per_partition}-word partitions.")

    replace = n_partitions > max_start
    starts  = ([rng.randint(0, max_start) for _ in range(n_partitions)]
               if replace else rng.sample(range(max_start + 1), n_partitions))

    return [
        {
            "partition_id": f"{book_name}_partition_{int_to_label(i)}",
            "book":         book_name,
            "genre":        genre,
            "start_token":  start,
            "text":         ' '.join(tokens[start: start + words_per_partition]),
            "word_count":   words_per_partition,
        }
        for i, start in enumerate(starts)
    ]

## Process All Books

In [4]:
rng         = random.Random(RANDOM_SEED)
all_records = []

for filepath in BOOK_FILES:
    name  = book_label(filepath)
    genre = GENRE_MAP.get(name, "unknown")
    tokens = load_and_clean(filepath)
    recs   = sample_partitions(tokens, name, genre, PARTITIONS_PER_BOOK, WORDS_PER_PARTITION, rng)
    all_records.extend(recs)
    print(f"{name} [{genre}]: {len(tokens):,} tokens → {len(recs)} partitions")

print(f"\nTotal partitions: {len(all_records)}")

AutobiographyOfBenjaminFranklin [biography]: 76,203 tokens → 200 partitions
LifeOfKingEdwardVII [biography]: 147,715 tokens → 200 partitions
NarrativeOfFrederickDouglass [biography]: 40,750 tokens → 200 partitions
StoryOfMyLife [biography]: 134,873 tokens → 200 partitions
AutobiographyOfCharlesDarwin [biography]: 22,684 tokens → 200 partitions
TheWonderfulWizardOfOz [fantasy]: 39,649 tokens → 200 partitions
AliceInWonderland [fantasy]: 26,525 tokens → 200 partitions
ThroughTheLookingGlass [fantasy]: 29,752 tokens → 200 partitions
GrimmsFairyTales [fantasy]: 101,111 tokens → 200 partitions
GulliversTravels [fantasy]: 105,079 tokens → 200 partitions
Frankenstein [horror]: 75,042 tokens → 200 partitions
CallOfCthulu [horror]: 11,968 tokens → 200 partitions
Dracula [horror]: 161,321 tokens → 200 partitions
TalesOfPoe [horror]: 95,043 tokens → 200 partitions
DrJekyllAndMrHyde [horror]: 25,631 tokens → 200 partitions
PrideAndPrejudice [romance]: 127,359 tokens → 200 partitions
RomeoAndJuliet

## Build the DataFrame

In [5]:
df = pd.DataFrame(all_records).set_index("partition_id")
print(f"Shape: {df.shape}")
df.head(6)

Shape: (5000, 5)


,book,genre,start_token,text,word_count
partition_id,,,,,
AutobiographyOfBenjaminFranklin_partition_a,AutobiographyOfBenjaminFranklin,biography,14592,Boston. I took leave of Keimer as going to see...,400
AutobiographyOfBenjaminFranklin_partition_b,AutobiographyOfBenjaminFranklin,biography,3278,translated into all the languages of Europe. I...,400
AutobiographyOfBenjaminFranklin_partition_c,AutobiographyOfBenjaminFranklin,biography,36048,"in conversation, which makes his company still...",400
AutobiographyOfBenjaminFranklin_partition_d,AutobiographyOfBenjaminFranklin,biography,32098,propose it to such as they thought lovers of r...,400
AutobiographyOfBenjaminFranklin_partition_e,AutobiographyOfBenjaminFranklin,biography,29256,deserting it one after another. [59] Recalled ...,400
AutobiographyOfBenjaminFranklin_partition_f,AutobiographyOfBenjaminFranklin,biography,18289,and repeated; Watson and Osborne gave up the c...,400


## Validation

In [6]:
assert (df["word_count"] == WORDS_PER_PARTITION).all(), "Word count mismatch!"
assert (df.groupby("book").size() == PARTITIONS_PER_BOOK).all(), "Partition count mismatch!"
assert df.index.is_unique, "Duplicate partition IDs!"
print("All validation checks passed.")

All validation checks passed.


## Genre Distribution

In [7]:
genre_counts = df.groupby("genre").size().rename("partition_count")
print(genre_counts.to_string())
print(f"\nTotal: {genre_counts.sum()}")

genre
biography    1000
fantasy      1000
horror       1000
romance      1000
sci-fi       1000

Total: 5000


## Export Partitions
Saves `partitions.csv` — the input consumed by `genre_classifier.ipynb`.

In [8]:
df.reset_index().to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(df)} partitions to '{OUTPUT_CSV}'")

Saved 5000 partitions to 'TestingData/partitions.csv'
